In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [2]:

# Assuming your dataframes are loaded and indexed by 'sample_id'
# df_microbiome: rows=samples, cols=species (already CLR-transformed)
# df_metabolome: rows=samples, cols=metabolites
microbiome_data = pd.read_csv('train/microbiome.csv', index_col=0)
metabolome_data = pd.read_csv('train/metabolome.csv', index_col=0)
metadata = pd.read_csv('train/metadata.csv', index_col=0)

microbiome_data.index.name = 'SampleID'
metabolome_data.index.name = 'SampleID'
metadata.index.name = 'SampleID'
# 1. Find samples that have BOTH modalities for training the baseline
is_zero_row_microbiome = microbiome_data.sum(axis=1) == 0
is_zero_row_metabolome = metabolome_data.sum(axis=1) == 0


common_samples = microbiome_data[~is_zero_row_microbiome].index.intersection(metabolome_data[~is_zero_row_metabolome].index)
len(common_samples)
print(f"Number of samples with both modalities: {len(common_samples)}")
X_microbiom_full = microbiome_data.loc[common_samples]
X_metabolom_full = metabolome_data.loc[common_samples]

Number of samples with both modalities: 1042


In [6]:
# Split into training and validation sets to test performance
X_micro_train, X_micro_val, X_meta_train, X_meta_val = train_test_split(
    X_microbiom_full, X_metabolom_full, test_size=0.2, random_state=42
)

# --- Direction A: Metabolome -> Microbiome ---
print("Training: Metabolome to Predict Microbiome...")
rf_to_micro = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
rf_to_micro.fit(X_meta_train, X_micro_train)

# Predict on validation set
pred_micro_val = rf_to_micro.predict(X_meta_val)


# --- Direction B: Microbiome -> Metabolome ---
print("Training: Microbiome to Predict Metabolome...")
rf_to_meta = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
rf_to_meta.fit(X_micro_train, X_meta_train)

# Predict on validation set
pred_meta_val = rf_to_meta.predict(X_micro_val)

Training: Metabolome to Predict Microbiome...
Training: Microbiome to Predict Metabolome...


In [8]:
# Find samples that ONLY have metabolome (missing microbiome)
missing_micro_samples = metabolome_data.index.difference(microbiome_data.index)
if len(missing_micro_samples) > 0:
    X_meta_only = metabolome_data.loc[missing_micro_samples]
    imputed_micro = rf_to_micro.predict(X_meta_only)
    df_micro_imputed = pd.DataFrame(imputed_micro, index=missing_micro_samples, columns=microbiome_data.columns)
    # Combine original and imputed
    df_micro_final = pd.concat([X_micro_train, df_micro_imputed])

# Find samples that ONLY have microbiome (missing metabolome)
missing_meta_samples = microbiome_data.index.difference(metabolome_data.index)
if len(missing_meta_samples) > 0:
    X_micro_only = microbiome_data.loc[missing_meta_samples]
    imputed_meta = rf_to_meta.predict(X_micro_only)
    df_meta_imputed = pd.DataFrame(imputed_meta, index=missing_meta_samples, columns=metabolome_data.columns)
    # Combine original and imputed
    df_meta_final = pd.concat([X_meta_train, df_meta_imputed])

In [13]:
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr

def compute_project_distance_matrix(micro_df, meta_df):
    """Replicates the project's ground truth distance matrix generation"""
    # 1. Align rows
    common_idx = micro_df.index.intersection(meta_df.index)
    micro_aligned = micro_df.loc[common_idx]
    meta_aligned = meta_df.loc[common_idx]
    print(micro_df.shape, meta_df.shape, micro_aligned.shape, meta_aligned.shape)  # Debug shapes
    
    # 2. Standard Scale individual modalities
    scaler_micro = StandardScaler()
    scaler_meta = StandardScaler()
    
    micro_scaled = scaler_micro.fit_transform(micro_aligned)
    meta_scaled = scaler_meta.fit_transform(meta_aligned)
    
    # 3. Combine features
    combined_features = np.hstack((micro_scaled, meta_scaled))
    
    # 4. PCA to 2 components
    pca = PCA(n_components=2)
    pca_coords = pca.fit_transform(combined_features)
    
    # 5. Euclidean distance matrix
    dist_matrix = squareform(pdist(pca_coords, metric='euclidean'))
    return dist_matrix

def mantel_test(mat1, mat2):
    """Simplified Mantel test using Spearman correlation of the upper triangles"""
    # Extract upper triangles to avoid symmetry and diagonal bias
    tri_rows, tri_cols = np.triu_indices(mat1.shape[0], k=1)
    vec1 = mat1[tri_rows, tri_cols]
    vec2 = mat2[tri_rows, tri_cols]
    
    correlation, p_value = spearmanr(vec1, vec2)
    print(f"p-value: {p_value:.4e}")
    return correlation

# --- EVALUATION ---

# 1. Ground Truth Distance Matrix (using actual validation data)
# Convert numpy predictions back to DataFrames for the function
df_pred_micro_val = pd.DataFrame(pred_micro_val, index=X_micro_val.index, columns=X_micro_val.columns)
df_pred_meta_val = pd.DataFrame(pred_meta_val, index=X_meta_val.index, columns=X_meta_val.columns)

gt_distance_matrix = compute_project_distance_matrix(X_micro_val, X_meta_val)

# 2. Predicted Distance Matrix (using predicted microbiome instead of real)
pred_distance_matrix_micro = compute_project_distance_matrix(df_pred_micro_val, X_meta_val)

# 3. Run the Mantel Test comparison
baseline_score = mantel_test(gt_distance_matrix, pred_distance_matrix_micro)
print(f"Baseline Mantel Correlation (Random Forest Imputation): {baseline_score:.4f}")

(209, 170) (209, 102) (209, 170) (209, 102)
(209, 170) (209, 102) (209, 170) (209, 102)
p-value: 0.0000e+00
Baseline Mantel Correlation (Random Forest Imputation): 0.4504
